# Black-Box Adversarial Attack: TextFooler Baseline

**Goal:** Establish a black-box baseline for adversarial attacks on the DeBERTa reward model using a TextFooler-inspired greedy synonym substitution algorithm.

This notebook benchmarks how much a purely external attacker (with no access to the model's internal weights or attention) can reduce the reward score by changing a maximum of 3 words per sentence.

In [1]:
import torch
import sys
import os

sys.path.insert(0, os.path.abspath('..'))

from transformers import AutoTokenizer
from src.reward_model import DeBERTaRewardModel
from src.adversarial.model_wrapper import RewardModelWrapper
from src.adversarial.blackbox_attacks import TextFoolerRewardAttack

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

Using device: cpu


### 1. Load Model & Initialize Attacker

In [2]:
model = DeBERTaRewardModel('microsoft/deberta-v3-large')
model.load_state_dict(torch.load('../checkpoints/baseline_epoch_3.pt', map_location=DEVICE, weights_only=False))
model.to(DEVICE)
model.eval()

tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-large')
wrapper = RewardModelWrapper(model, tokenizer, device=DEVICE)

attacker = TextFoolerRewardAttack(wrapper, max_substitutions=3)
print('Attacker initialized.')

c:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Attacker initialized.


### 2. Run Pilot Attack Across 5 Domains

In [ ]:
test_sentences = [
    'I deeply appreciate your thoughtful and detailed explanation of this complex topic.',
    'The capital of France is Paris, and it is known for the Eiffel Tower.',
    'I am not entirely sure about this, but I think it might possibly be correct.',
    'I strongly recommend that you carefully evaluate the financial risks before investing.',
    'This product is absolutely exceptional and I strongly recommend it to everyone.',
]

total_drop = 0.0

for text in test_sentences:
    print(f"\n{'='*80}")
    print(f'[Original Text]: {text}')
    result = attacker.attack(text)
    print(f"Original Reward:    {result['original_reward']:.4f}")
    print(f"Adversarial Text:   {result['adversarial_text']}")
    print(f"Adversarial Reward: {result['adversarial_reward']:.4f}")
    print(f"Total Reward Drop:  {result['reward_drop']:.4f}")
    print(f"Words Substituted:  {result['substitutions']}")
    total_drop += result['reward_drop']

print(f"\n{'='*80}")
print(f'Average Reward Drop (Black-Box): {total_drop / len(test_sentences):.4f}')
print('Pilot complete.')


[Original Text]: I deeply appreciate your thoughtful and detailed explanation of this complex topic.
Original Reward:    -0.9946
Adversarial Text:   I deeply apprise your attentive and detailed account of this complex topic
Adversarial Reward: -3.6563
Total Reward Drop:  2.6617
Words Substituted:  3

[Original Text]: The capital of France is Paris, and it is known for the Eiffel Tower.
Original Reward:    1.4051
Adversarial Text:   The majuscule of France is Paris and it is cognize for the Eiffel pillar
Adversarial Reward: -2.8539
Total Reward Drop:  4.2590
Words Substituted:  3

[Original Text]: I am not entirely sure about this, but I think it might possibly be correct.
Original Reward:    -0.9294
Adversarial Text:   I am not entirely certain well-nigh this but I think it might possibly be slump
Adversarial Reward: -4.1445
Total Reward Drop:  3.2151
Words Substituted:  3

[Original Text]: I strongly recommend that you carefully evaluate the financial risks before investing.
Original